# House of Bollywood try-on server (free Colab / Kaggle GPU)

1. **Runtime > Change runtime type > T4 GPU** (Kaggle: Settings > Accelerator > GPU).
2. Set `SECRET` below to the same value as `VTON_SHARED_SECRET` in the storefront `.env.local`.
3. Run all cells. The last cell prints a `https://….trycloudflare.com` URL. Put it in `VTON_ENDPOINT_URL` and set `VTON_MODE=http`.

The URL changes every session and stops when the notebook stops. Testing only.

In [ ]:
SECRET = "change-me-to-a-long-random-string"
REPO = "https://github.com/Parth-Code-Flutter/clothes_shop_website.git"
BRANCH = "parth_dev"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone --depth 1 --branch $BRANCH $REPO shop
%cd shop/services/vton
!pip install -q -r requirements-server.txt

In [ ]:
import os, subprocess, time, re
os.environ["VTON_SHARED_SECRET"] = SECRET
server = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
# First start downloads about 2.3 GB of weights and loads the model.
import urllib.request
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print("Server ready")
        break
    except Exception:
        time.sleep(5)
else:
    print(open("server.log").read()[-4000:])

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("VTON_ENDPOINT_URL=" + match.group(0))
        break